# Sesión 2: Hands-on - Tu primer LLM

**Curso:** Introducción a Large Language Models  
**Profesor:** Francisco Pérez Carbajal  
**Semana:** 1 (Fundamentos)  

---

## Objetivos de hoy

1. ✅ Cargar y usar un LLM pre-entrenado
2. ✅ Experimentar con parámetros de generación (temperature, top-k, top-p)
3. ✅ Hacer **conditional generation** (prompting)
4. ✅ Conectar con ideas para tu proyecto

**Formato:** Ejercicios progresivos con experimentación

## Setup

Primero, instalar las bibliotecas necesarias:

In [ ]:
!pip install transformers torch -q

In [ ]:
import transformers
import torch
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

---

# Ejercicio 1: Cargar tu primer LLM

## Objetivo
Desmitificar el proceso de cargar y usar un modelo de lenguaje.

## Paso 1.1: El pipeline más simple

Hugging Face `transformers` tiene una interfaz súper simple llamada **pipeline**.

In [ ]:
from transformers import pipeline

# Crear un generador de texto
generator = pipeline('text-generation', model='gpt2')

print("✓ Modelo cargado correctamente")

### Primer generación de texto

In [ ]:
# Tu primer texto generado
prompt = "Once upon a time"

result = generator(prompt, max_length=50, num_return_sequences=1)

print("Prompt:", prompt)
print("\nTexto generado:")
print(result[0]['generated_text'])

### 🎯 Tu turno

Prueba diferentes prompts. Algunos ejemplos:
- "The future of artificial intelligence"
- "In a world where"
- "The secret to happiness is"

**Ejecuta la celda varias veces** con el mismo prompt. ¿Genera siempre lo mismo?

In [ ]:
# TU CÓDIGO AQUÍ
prompt = ""  # Cambia esto

result = generator(prompt, max_length=50)
print(result[0]['generated_text'])

## Paso 1.2: ¿Qué está pasando "bajo el capó"?

El `pipeline` es conveniente, pero esconde mucho. Veamos el proceso completo:

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Cargar modelo y tokenizer por separado
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

print("✓ Modelo y tokenizer cargados")

### Paso a paso: De texto a generación

#### 1️⃣ Texto → Tokens (números)

In [ ]:
prompt = "The cat sat on"

# Tokenizar
input_ids = tokenizer.encode(prompt, return_tensors='pt')

print(f"Texto: '{prompt}'")
print(f"Token IDs: {input_ids.tolist()[0]}")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(input_ids[0])}")

#### 2️⃣ Modelo genera tokens (uno a la vez)

In [ ]:
# Generar los siguientes 10 tokens
output = model.generate(
    input_ids,
    max_new_tokens=10,
    do_sample=True,  # Muestreo estocástico
    pad_token_id=tokenizer.eos_token_id
)

# Decodificar
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print(f"\nTexto generado:\n{generated_text}")

### 📊 Comparación visual

Generemos **múltiples** completaciones para ver la variabilidad:

In [ ]:
prompt = "The cat sat on"
input_ids = tokenizer.encode(prompt, return_tensors='pt')

# Generar 5 diferentes completaciones
outputs = model.generate(
    input_ids,
    max_new_tokens=15,
    num_return_sequences=5,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print(f"Prompt: '{prompt}'\n")
for i, output in enumerate(outputs, 1):
    text = tokenizer.decode(output, skip_special_tokens=True)
    print(f"{i}. {text}")

**Pregunta:** ¿Por qué cada generación es diferente si usamos el mismo prompt?

<details>
<summary>💡 Respuesta (click para ver)</summary>
    
El modelo **muestrea** de la distribución de probabilidad. No siempre elige el token más probable, sino que introduce aleatoriedad controlada.
</details>

---

# Ejercicio 2: Parámetros de generación

## Objetivo
Entender cómo controlar el "estilo" de la generación.

## 2.1: Temperature (T)

**Temperature** controla cuánta "aleatoriedad" hay en la generación:

- **T = 0**: Determinista (siempre elige el más probable)
- **T = 1**: "Normal" (muestreo según probabilidades del modelo)
- **T > 1**: Más aleatorio/creativo/loco
- **T → ∞**: Completamente aleatorio

**Intuición matemática:**
$$p_T(x_i | x_{1:i-1}) \propto p(x_i | x_{1:i-1})^{1/T}$$

In [ ]:
def generate_with_temperature(prompt, temperature, num_sequences=3):
    """
    Genera texto con una temperature específica.
    """
    results = generator(
        prompt,
        max_length=30,
        temperature=temperature,
        num_return_sequences=num_sequences,
        do_sample=True if temperature > 0 else False
    )
    
    return [r['generated_text'] for r in results]

### Experimento: Comparar diferentes temperatures

In [ ]:
prompt = "Artificial intelligence is"

for temp in [0.1, 0.7, 1.5]:
    print(f"\n{'='*60}")
    print(f"Temperature = {temp}")
    print(f"{'='*60}")
    
    results = generate_with_temperature(prompt, temp, num_sequences=3)
    
    for i, text in enumerate(results, 1):
        print(f"\n{i}. {text}")

### 🎯 Tu turno: Encuentra la temperature óptima

**Tarea:** Para cada objetivo, encuentra la mejor temperature:

1. **Escribir código Python** → Temperature = ?
2. **Escribir poesía creativa** → Temperature = ?
3. **Responder preguntas factuales** → Temperature = ?

In [ ]:
# Experimento 1: Código Python
prompt = "def calculate_factorial(n):\n    "
temperature = 0.0  # Ajusta esto

results = generate_with_temperature(prompt, temperature, num_sequences=3)
for i, text in enumerate(results, 1):
    print(f"\n{i}.\n{text}")

In [ ]:
# Experimento 2: Poesía
prompt = "Roses are red, violets are blue,"
temperature = 1.0  # Ajusta esto

results = generate_with_temperature(prompt, temperature, num_sequences=3)
for i, text in enumerate(results, 1):
    print(f"\n{i}. {text}")

In [ ]:
# Experimento 3: Pregunta factual
prompt = "The capital of France is"
temperature = 0.0  # Ajusta esto

results = generate_with_temperature(prompt, temperature, num_sequences=3)
for i, text in enumerate(results, 1):
    print(f"\n{i}. {text}")

## 2.2: Top-k Sampling

**Top-k**: Solo considera los **k tokens más probables** en cada paso.

- k = 1: Igual que T=0 (greedy)
- k = 50: Balance razonable
- k = vocabulario completo: Sin restricción

In [ ]:
prompt = "The most important invention in history is"

for k in [1, 10, 50]:
    print(f"\n{'='*60}")
    print(f"Top-k = {k}")
    print(f"{'='*60}")
    
    results = generator(
        prompt,
        max_length=40,
        do_sample=True,
        top_k=k,
        num_return_sequences=3
    )
    
    for i, r in enumerate(results, 1):
        print(f"\n{i}. {r['generated_text']}")

## 2.3: Top-p Sampling (Nucleus Sampling)

**Top-p**: Selecciona tokens hasta que la probabilidad acumulada llegue a **p**.

- p = 0.1: Muy conservador
- p = 0.9: Balance común
- p = 1.0: Sin restricción

In [ ]:
prompt = "In the year 2050,"

for p in [0.5, 0.9, 0.95]:
    print(f"\n{'='*60}")
    print(f"Top-p = {p}")
    print(f"{'='*60}")
    
    results = generator(
        prompt,
        max_length=50,
        do_sample=True,
        top_p=p,
        top_k=0,  # Deshabilitar top-k
        num_return_sequences=2
    )
    
    for i, r in enumerate(results, 1):
        print(f"\n{i}. {r['generated_text']}")

### 🎯 Tu turno: Experimenta con combinaciones

Prueba diferentes combinaciones de parámetros:

In [ ]:
# TU EXPERIMENTO AQUÍ
prompt = ""  # Tu prompt

results = generator(
    prompt,
    max_length=60,
    temperature=1.0,    # Ajusta
    top_k=50,          # Ajusta
    top_p=0.9,         # Ajusta
    num_return_sequences=3
)

for i, r in enumerate(results, 1):
    print(f"\n{i}. {r['generated_text']}")

---

# Ejercicio 3: Conditional Generation y Prompting

## Objetivo
Usar el **prompt** como la interfaz para "programar" el LLM.

## 3.1: Question Answering

In [ ]:
# Formato Q&A simple
prompt = "Q: What is the capital of France?\nA:"

result = generator(
    prompt,
    max_length=30,
    temperature=0.1,
    num_return_sequences=1
)

print(result[0]['generated_text'])

## 3.2: Few-shot Learning

**Idea clave:** Dar ejemplos en el prompt para "enseñar" al modelo la tarea.

In [ ]:
# Traducción español → inglés con ejemplos
prompt = """Spanish: Hola
English: Hello

Spanish: Gracias
English: Thank you

Spanish: Buenos días
English:"""

result = generator(
    prompt,
    max_length=len(tokenizer.encode(prompt)) + 5,
    temperature=0.1,
    num_return_sequences=1
)

print(result[0]['generated_text'])

### 🎯 Tu turno: Diseña tu propio few-shot prompt

Elige una tarea:
1. Clasificación de sentimientos (positivo/negativo)
2. Traducción
3. Extracción de información
4. Otra idea

In [ ]:
# Ejemplo: Clasificación de sentimientos
prompt = """Text: I love this movie!
Sentiment: Positive

Text: This is terrible.
Sentiment: Negative

Text: Not sure how I feel about this.
Sentiment:"""

# TU CÓDIGO AQUÍ


## 3.3: Code Generation

In [ ]:
# Generar código Python
prompt = """# Function to check if a number is prime
def is_prime(n):
    """

result = generator(
    prompt,
    max_length=100,
    temperature=0.2,
    num_return_sequences=1
)

print(result[0]['generated_text'])

## 3.4: Creative Writing

In [ ]:
# Generar un cuento corto
prompt = """Write a short story about a robot who learns to feel emotions.

Once upon a time, in a laboratory far away,"""

result = generator(
    prompt,
    max_length=150,
    temperature=0.8,
    top_p=0.9,
    num_return_sequences=1
)

print(result[0]['generated_text'])

---

# Challenge Final: Proyecto personal

## 🎯 Tu tarea

Elige **UNA** de las siguientes tareas y diseña el mejor prompt + parámetros:

### Opción A: Traducción español ↔ inglés
- Usa few-shot learning
- Prueba diferentes números de ejemplos (2, 3, 5)
- Encuentra la mejor temperature

### Opción B: Generador de código Python
- Descripción en lenguaje natural → código funcional
- Experimenta con docstrings como guía
- Temperature baja para código más correcto

### Opción C: Creative writing
- Cuento corto en un género específico (sci-fi, fantasy, etc.)
- Experimenta con diferentes temperatures
- Prueba dar ejemplos de estilo

### Opción D: Tu propia idea
¡Sorpréndenos!

In [ ]:
# TU PROYECTO AQUÍ

prompt = """
# Diseña tu prompt aquí
"""

result = generator(
    prompt,
    max_length=100,           # Ajusta
    temperature=0.7,          # Ajusta
    top_p=0.9,               # Ajusta
    num_return_sequences=3   # Ajusta
)

for i, r in enumerate(results, 1):
    print(f"\n{'='*60}")
    print(f"Resultado {i}:")
    print(f"{'='*60}")
    print(r['generated_text'])

---

# Reflexión final

## ¿Qué aprendimos hoy?

1. ✅ **Cargar modelos** es fácil con Hugging Face
2. ✅ **Temperature, top-k, top-p** controlan el estilo de generación
3. ✅ **Prompting** es el arte de "programar" LLMs
4. ✅ **Few-shot learning** permite enseñar tareas sin fine-tuning

## Limitaciones que observamos

- GPT-2 es "viejo" (2019) y pequeño (124M params)
- A veces genera respuestas incorrectas o sin sentido
- Funciona mejor en inglés que en español
- No puede acceder a información actualizada

## Próximos pasos

**Semana 2:**
- Entender la arquitectura Transformer
- ¿Cómo se entrenan estos modelos?
- Modelos más modernos (Llama, Mistral, etc.)

**Para tu proyecto:**
- Pensar en qué problema quieres resolver
- ¿Necesitas fine-tuning o basta con prompting?
- ¿Qué datos necesitarías?

---

## 📚 Recursos adicionales

### Documentación:
- [Hugging Face Transformers](https://huggingface.co/docs/transformers/)
- [Generation strategies](https://huggingface.co/docs/transformers/main/en/generation_strategies)

### Tutoriales:
- [The Illustrated GPT-2](https://jalammar.github.io/illustrated-gpt2/)
- [Hugging Face NLP Course](https://huggingface.co/learn/nlp-course/)

### Para explorar:
- [Modelos en Hugging Face Hub](https://huggingface.co/models)
- [Datasets para NLP](https://huggingface.co/datasets)